# Fine-tuning Generator — Qwen2.5-7B + Unsloth + QLoRA

Ce notebook fine-tune **Qwen2.5-7B-Instruct** sur `generator_dataset.json`
(produit par `scripts/prepare_dataset.py`) pour générer des commentaires
Instagram dans le registre des T-types validés humainement.

**Cible matérielle** : RunPod / Google Colab Pro avec **GPU A100 40 GB**
(ou A6000 / RTX 4090 24 GB en réduisant `BATCH_SIZE` à 2 et
`MAX_SEQ_LENGTH` à 384). Sur A100, un epoch sur ~500 paires prend ~3 min.

**Pipeline complet** :
1. Préparation du dataset locale → `python scripts/prepare_dataset.py`
2. Upload `scripts/data/generator_dataset.json` à côté de ce notebook
3. Exécuter ce notebook → modèle fine-tuné poussé sur HuggingFace
4. Conversion GGUF → import dans Ollama sur le Mac Mini (cf. dernière cellule)

> **Important** : le format Alpaca utilisé ici (`### Instruction / ### Input / ### Response`)
> doit être **strictement identique** entre l'entraînement et l'inférence.
> Toute divergence (espaces, casse, sauts de ligne) dégrade la qualité.

## 1. Installation

`unsloth` embarque déjà une version compatible de `transformers`, `peft`,
`bitsandbytes`, `xformers` et `triton` — on installe ensuite `trl` et
`accelerate` au cas où l'environnement Colab/RunPod ne les fournit pas.

> Sur **Google Colab**, redémarrer le runtime après cette cellule
> (Runtime → Restart) si une dépendance refuse de se charger.

In [ ]:
!pip install unsloth
!pip install trl transformers accelerate bitsandbytes

import torch
assert torch.cuda.is_available(), "GPU CUDA introuvable — bascule sur un runtime A100/T4."
print(f"GPU détecté : {torch.cuda.get_device_name(0)}")
print(f"VRAM dispo : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")

## 2. Configuration

Hyper-paramètres de référence (A100 40 GB, dataset ~500-2000 paires).

- `LORA_R=16` / `LORA_ALPHA=32` → ratio standard 1:2, expressivité raisonnable sans
  surapprentissage rapide.
- `LEARNING_RATE=2e-4` avec scheduler `cosine` + `warmup_ratio=0.1` → recette
  Unsloth/QLoRA éprouvée.
- Effective batch size = `BATCH_SIZE × GRAD_ACCUMULATION` = **16**.
- 3 epochs : suffisant pour un dataset court (>200 paires) sans overfit.

> Adapter `HF_REPO` à ton compte HuggingFace avant exécution. `HF_TOKEN`
> doit être disponible (variable d'env ou `notebook_login()`).

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "./qwen_generator_finetuned"
DATASET_PATH = "./generator_dataset.json"
HF_REPO = "ton-username/aitertainment-generator"

MAX_SEQ_LENGTH = 512
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
EPOCHS = 3
LEARNING_RATE = 2e-4

print(f"Modèle de base : {MODEL_NAME}")
print(f"Dataset       : {DATASET_PATH}")
print(f"Sortie        : {OUTPUT_DIR}")
print(f"Repo HF       : {HF_REPO}")
print(
    f"Effective batch size : {BATCH_SIZE * GRAD_ACCUMULATION} "
    f"(BATCH_SIZE={BATCH_SIZE} × GRAD_ACCUMULATION={GRAD_ACCUMULATION})"
)

## 3. Chargement du modèle Qwen2.5-7B + injection des adapters LoRA

Unsloth charge le modèle en **4-bit** (NF4 quantization) puis injecte des
adapters LoRA sur les 7 modules d'attention/MLP de chaque bloc transformer.
Seuls les ~40 M paramètres LoRA sont entraînés (vs 7 B en plein), ce qui
permet le fine-tuning sur 24-40 Go de VRAM.

`use_gradient_checkpointing=True` recalcule les activations forward pour
réduire la VRAM (×2 sur la mémoire activations, coût : ~20 % de temps).

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing=True,
)

model.print_trainable_parameters()

## 4. Chargement du dataset et formatage Alpaca

Le dataset est une liste JSON top-level d'objets `{instruction, input, output}`
produite par `scripts/prepare_dataset.py`. On la convertit au format texte
brut Alpaca, qui est ce que `SFTTrainer` consomme via `dataset_text_field`.

> **Le template Alpaca utilisé ici doit être réutilisé tel quel à
> l'inférence** (cf. cellule 7) — sinon le modèle ne reconnaît pas le prompt.

In [ ]:
import json
from datasets import Dataset

with open(DATASET_PATH) as f:
    data = json.load(f)

print(f"Paires chargées : {len(data)}")
assert len(data) >= 50, (
    f"Dataset trop petit ({len(data)} paires) — collecte plus de données via "
    "le bot Telegram (validations) avant fine-tuning."
)


def format_alpaca(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    }


dataset = Dataset.from_list(data)
dataset = dataset.map(format_alpaca)

print("\nExemple formaté :")
print(dataset[0]["text"])
print(f"\nNombre de tokens (approx) sur l'exemple 0 : "
      f"{len(tokenizer.encode(dataset[0]['text']))}")

## 5. Entraînement (SFTTrainer)

`fp16=True` est correct sur A100/T4/V100 ; sur Ampere/Hopper récents
(A100, H100), `bf16=True` est plus stable — décommenter au besoin.

**Durée indicative** sur A100 40 GB :
- 200 paires × 3 epochs ≈ **2 min**
- 1 000 paires × 3 epochs ≈ **8 min**
- 5 000 paires × 3 epochs ≈ **35 min**

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION,
        learning_rate=LEARNING_RATE,
        fp16=True,
        # bf16=True,  # à privilégier sur A100/H100
        logging_steps=10,
        save_steps=100,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
    ),
)

trainer_stats = trainer.train()
print("\nEntraînement terminé.")
print(f"Loss finale : {trainer_stats.training_loss:.4f}")

## 6. Sauvegarde locale + push HuggingFace

`save_pretrained` sauvegarde uniquement les **adapters LoRA** (~150 Mo)
plus le tokenizer — pas les 14 Go du modèle de base. C'est ce qu'on push
sur HF, ce qui économise bande passante et stockage.

> Côté inférence (Ollama / llama.cpp), il faudra **merger les adapters**
> dans le base model pour produire un `.gguf` autonome (cf. cellule
> "Intégration Mac Mini").

**Authentification HuggingFace** : un `HF_TOKEN` avec scope `write` doit
être disponible — sur Colab, exécuter `from huggingface_hub import notebook_login; notebook_login()`
en amont. Sur RunPod, exporter `HF_TOKEN` en variable d'env.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapters LoRA + tokenizer sauvegardés : {OUTPUT_DIR}")

from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True, private=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO,
    repo_type="model",
)
print(f"Modèle pushé sur HuggingFace : https://huggingface.co/{HF_REPO}")

## 7. Test rapide d'inférence

Sanity check immédiat sur le modèle fraîchement entraîné. Le prompt utilise
**exactement** le template Alpaca de l'entraînement, terminé par `### Response:`
suivi d'un saut de ligne — le modèle doit compléter avec un commentaire
respectant les règles absolues du system prompt (≤ 8 mots, minuscule, pas
de mots interdits).

`FastLanguageModel.for_inference(model)` active le mode optim Unsloth
(2× plus rapide en génération vs mode train).

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

test_input = """### Instruction:
Tu es un utilisateur Instagram. Génère un commentaire naturel et humain pour ce Reel.

### Input:
T-type: T2
Niche: humour
Caption: quand tu rates ton exam mais que t'assumes
Hashtags: #humour #étudiant
Audio: 12345

### Response:
"""

inputs = tokenizer(test_input, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
)
generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Commentaire généré :")
print(generated.strip())

## 8. Intégration Mac Mini (Ollama)

Pour utiliser le modèle fine-tuné dans le pipeline AItertainment, il faut le
**convertir au format GGUF** puis l'**importer dans Ollama**. Cinq étapes,
toutes exécutées sur le Mac Mini M4 (sauf 1 et 2 qui peuvent rester sur la
machine A100 si on veut éviter de télécharger 14 Go).

### Étape 1 — Merger les adapters LoRA dans le modèle de base

Les adapters seuls (~150 Mo poussés sur HF) ne suffisent pas : `llama.cpp`
attend un modèle complet. On fusionne LoRA + base en un nouveau dossier.

```python
# À exécuter en fin de notebook (avant de fermer la session GPU)
# ou dans un script séparé après pull du repo HF.
from unsloth import FastLanguageModel

model.save_pretrained_merged(
    "./qwen_generator_merged",
    tokenizer,
    save_method="merged_16bit",  # FP16 — meilleure qualité avant quantization GGUF
)
```

### Étape 2 — Conversion GGUF avec `llama.cpp`

```bash
# Sur le Mac Mini (ou la machine GPU avant download)
brew install cmake
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
make -j

# Convertit le modèle merged en GGUF FP16 (~14 Go)
python convert_hf_to_gguf.py ../qwen_generator_merged \
  --outfile ../aitertainment-generator-f16.gguf \
  --outtype f16

# Quantize en Q4_K_M (~4.5 Go, recommandé pour Apple Silicon)
./quantize ../aitertainment-generator-f16.gguf \
  ../aitertainment-generator-q4km.gguf Q4_K_M
```

Variantes utiles :
- `Q5_K_M` (~5 Go) : meilleure qualité, ~10 % plus lent.
- `Q3_K_M` (~3.5 Go) : plus rapide, dégrade un peu le registre.

### Étape 3 — Créer un `Modelfile` Ollama

Créer `~/aitertainment/Modelfile` :

```
FROM ./aitertainment-generator-q4km.gguf

# Le system prompt du brief (modules/classifier.py:GENERATE_COMMENTS_SYSTEM)
SYSTEM """Tu es un utilisateur lambda qui commente des Reels Instagram.
Tu ne représentes aucune marque. Tu écris comme tu parlerais à un ami — vite, sans réfléchir, en 3 secondes.

RÈGLES ABSOLUES (violation = réponse rejetée) :
- Maximum 8 mots par commentaire
- Minuscule en début (sauf nom propre)
- 0 ou 1 emoji max, placé naturellement
- Jamais ces mots : incroyable, impressionnant, vraiment, tellement, félicitations, magnifique, superbe, excellent, contenu, créateur, vidéo, post
- Pas de point final
- Registre oral SMS, pas rédactionnel
- Imparfait orthographiquement si ça sonne plus naturel

Retourne UNIQUEMENT ce JSON, rien d'autre :
{\"comments\": [\"commentaire 1\", \"commentaire 2\", \"commentaire 3\"]}"""

# Format Alpaca strict : doit matcher EXACTEMENT le template du notebook (cellule 4 + 7).
TEMPLATE """### Instruction:
{{ .System }}

### Input:
{{ .Prompt }}

### Response:
"""

PARAMETER temperature 0.8
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER num_predict 80
PARAMETER stop "### Instruction:"
PARAMETER stop "### Input:"
```

### Étape 4 — Importer dans Ollama

```bash
ollama create aitertainment-generator -f ~/aitertainment/Modelfile
ollama list   # vérifier que le modèle apparaît
ollama run aitertainment-generator "T-type: T2\nNiche: humour\nCaption: test"
```

### Étape 5 — Mettre à jour `.env` du projet

Dans `c:\\Users\\paulm\\Documents\\ASUS\\AItertainment\\aitertainment\\.env` (Mac Mini : `~/aitertainment/.env`) :

```bash
# Avant
OLLAMA_MODEL=qwen2.5:7b

# Après
OLLAMA_MODEL=aitertainment-generator
```

`modules/classifier.generate_comments` lit `config.OLLAMA_MODEL` à chaque
appel — aucun redémarrage du watcher n'est nécessaire pour des nouvelles
exécutions de scoring/génération. Pour les daemons `launchd` déjà chargés
(watcher, telegram_discovery_bot), un `launchctl unload` + `launchctl load`
recharge la conf .env.

### Vérification finale

```bash
# Le system prompt et le template doivent être bons.
ollama show aitertainment-generator --modelfile

# Smoke test depuis le projet :
cd ~/aitertainment
source .venv/bin/activate
python -c "from modules.classifier import generate_comments; \
  print(generate_comments({'type': 'T2'}, ['mdr', 'top'], niche='humour', \
    video_context={'caption': 'test', 'hashtags': ['humour'], 'audio_id': '1'}))"
```

Si la sortie respecte les règles absolues (≤ 8 mots, pas de mot interdit,
minuscule, pas de point final) → le fine-tuning a réussi à imprégner le
registre. Sinon, augmenter `EPOCHS` à 5 et collecter plus de données via le
bot Telegram (validations humaines).